# 🏛️ Huấn luyện Mô hình PhoBERT-CRF trên Google Colab

Notebook này tự động mount **Google Drive**, kiểm tra thư mục dự án `VPhoBertTagger`, nạp bộ dữ liệu **PAP_NER** và thực hiện huấn luyện mô hình **PhoBERT + CRF** với siêu tham số chuẩn từ bài báo (*PLOS ONE* - Dev F1: **97.82%**).

### 📌 Cấu hình huấn luyện tối ưu:
- **Backbone:** `vinai/phobert-base` (Contextual Embedding 768-dim)
- **Classifier Head:** `Linear(768 -> 11)` + `CRF` (Viterbi Decoding)
- **PhoBERT Learning Rate:** `2e-5` | **CRF Learning Rate:** `1e-3`
- **Scheduler:** Linear decay + 10% Warmup | **Batch size:** `32` | **Epochs:** `20` (Early stop: `3`)

## Bước 1: Kiểm tra GPU Google Colab
Đảm bảo Colab đang bật GPU (khuyên dùng T4, V100 hoặc A100).

In [1]:
!nvidia-smi

Sun Sep  6 12:05:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             68W /   70W |    7501MiB /  15360MiB |     44%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bước 2: Mount Google Drive & Tự động Định vị Thư mục `VPhoBertTagger`

In [2]:
# 1. Mount Google Drive
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    print('✅ Đã kết nối Google Drive thành công!')
except Exception as e:
    print(f'ℹ️ Ghi chú khi mount Drive: {e}')

# 2. Tìm kiếm thư mục VPhoBertTagger trên Google Drive
candidate_paths = [
    '/content/drive/MyDrive/VPhoBertTagger',
    '/content/drive/MyDrive/PHAPLUAT/VPhoBertTagger',
    '/content/drive/MyDrive/phapluat/VPhoBertTagger'
]

project_dir = None
for p in candidate_paths:
    if os.path.exists(p) and os.path.exists(os.path.join(p, 'main.py')):
        project_dir = p
        break

if project_dir is None:
    print('🔍 Đang tự động quét tìm thư mục VPhoBertTagger trong Drive...')
    search_root = '/content/drive/MyDrive'
    if os.path.exists(search_root):
        for root, dirs, files in os.walk(search_root):
            if 'VPhoBertTagger' in dirs and os.path.exists(os.path.join(root, 'VPhoBertTagger', 'main.py')):
                project_dir = os.path.join(root, 'VPhoBertTagger')
                break
            if root.count(os.sep) - search_root.count(os.sep) >= 3:
                dirs.clear()

if project_dir and os.path.exists(project_dir):
    os.chdir(project_dir)
    try:
        get_ipython().run_line_magic('cd', project_dir)
    except Exception:
        pass
    print(f'\n✅ ĐÃ ĐỊNH VỊ VÀ CHUYỂN VÀO THƯ MỤC DỰ ÁN:\n👉 {os.getcwd()}')
else:
    raise FileNotFoundError('❌ Không tìm thấy thư mục VPhoBertTagger trên Google Drive! Vui lòng kiểm tra lại vị trí đặt thư mục trên Drive.')

# Thêm vào PYTHONPATH
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Liệt kê các file trong thư mục dự án
!ls -la


✅ Đã kết nối Google Drive thành công!
/content

✅ ĐÃ ĐỊNH VỊ VÀ CHUYỂN VÀO THƯ MỤC DỰ ÁN:
👉 /content/drive/MyDrive/VPhoBertTagger
total 122
-rw------- 1 root root     0 Sep  6 11:08 '=0.7.2'
-rw------- 1 root root     0 Sep  6 11:08 '=4.28.0'
-rw------- 1 root root 12416 Sep  6 09:09  baseline.ipynb
-rw------- 1 root root  6217 Sep  6 08:32  cachhoatdong.md
-rw------- 1 root root  7449 Sep  6 08:39 'cách xây dựng và hoạt động của mô hình PhoBERT-CRF.md'
-rw------- 1 root root 11690 Sep  6 07:54  CRF.md
drwx------ 3 root root  4096 Sep  6 11:22  Dataset
-rw------- 1 root root  6556 Sep  6 08:36 'Dual-engine design.md'
-rw------- 1 root root  5586 Sep  6 08:42 'Experiments and results.md'
drwx------ 2 root root  4096 Sep  6 09:12  .git
-rw------- 1 root root    51 Sep  6 08:03  .ignore
drwx------ 2 root root  4096 Sep  6 09:12  img
-rw------- 1 root root  1189 Sep  6 07:47  inputs.md
-rw------- 1 root root   576 Sep  6 08:03  install_dependencies.sh
-rw------- 1 root root  156

## Bước 3: Cài đặt Thư viện Phụ thuộc (Dependencies)

In [3]:
!pip install -q transformers>=4.28.0 pytorch-crf>=0.7.2 prettytable tensorboardX vncorenlp
!pip install -q -r requirements.txt
print("\n✅ Cài đặt thư viện hoàn tất!")


✅ Cài đặt thư viện hoàn tất!


## Bước 4: Kiểm tra Dữ liệu PAP_NER (`Dataset/`)
Xác nhận sự tồn tại của `train_data.txt`, `dev_data.txt`, `test_data.txt`.

In [4]:
dataset_dir = Path("Dataset")
assert (dataset_dir / "train_data.txt").exists(), "Thiếu file train_data.txt trong Dataset/!"
assert (dataset_dir / "dev_data.txt").exists(), "Thiếu file dev_data.txt trong Dataset/!"
assert (dataset_dir / "test_data.txt").exists(), "Thiếu file test_data.txt trong Dataset/!"

print("✅ Cả 3 file dữ liệu PAP_NER đã sẵn sàng!")
!ls -lh Dataset/*.txt

print("\n--- 5 DÒNG MẪU ĐẦU TIÊN TỪ DEV_DATA.TXT ---")
with open('Dataset/dev_data.txt', 'r', encoding='utf-8') as f:
    for _ in range(5):
        print(f.readline().strip())

✅ Cả 3 file dữ liệu PAP_NER đã sẵn sàng!
-rw------- 1 root root 4.9M Sep  5 16:31 Dataset/dev_data.txt
-rw------- 1 root root 4.8M Sep  5 16:31 Dataset/test_data.txt
-rw------- 1 root root  62M Sep  5 16:31 Dataset/train_data.txt

--- 5 DÒNG MẪU ĐẦU TIÊN TỪ DEV_DATA.TXT ---
Đăng_ký N B-NP O O
thành_lập V B-VP O O
doanh_nghiệp V I-VP B-ĐT O
tư_nhân A O I-ĐT O
do E B-PP O O


## Bước 5: Mở TensorBoard Theo dõi Quá trình Huấn luyện

In [5]:
%load_ext tensorboard
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 3750), started 0:57:17 ago. (Use '!kill 3750' to kill it.)

<IPython.core.display.Javascript object>

## Bước 6: Thực hiện Huấn luyện Mô hình PhoBERT-CRF (Fine-Tuning)
Toàn bộ siêu tham số chuẩn từ bài báo (*PLOS ONE*):
- Model: `vinai/phobert-base` + `crf`
- PhoBERT LR: `2e-5` | CRF LR: `1e-3`
- Batch Size: `32` | Epochs: `20` | Early stop: `3` epochs

In [ ]:
!python main.py train \
    --task pap_ner \
    --data_dir ./Dataset \
    --model_name_or_path vinai/phobert-base \
    --model_arch crf \
    --output_dir outputs \
    --max_seq_length 256 \
    --train_batch_size 16 \
    --eval_batch_size 16 \
    --learning_rate 2e-5 \
    --classifier_learning_rate 1e-3 \
    --epochs 20 \
    --early_stop 3 \
    --overwrite_data \
    --run_test

Start TRAIN process...
Arguments: Namespace(type='train', task='pap_ner', data_dir='./Dataset', overwrite_data=True, load_weights=None, model_name_or_path='vinai/phobert-base', model_arch='crf', output_dir='outputs', max_seq_length=256, train_batch_size=16, eval_batch_size=16, learning_rate=2e-05, classifier_learning_rate=0.001, epochs=20, warmup_proportion=0.1, weight_decay=0.01, adam_epsilon=1e-06, max_grad_norm=1.0, early_stop=3.0, run_test=True, no_cuda=False, seed=42, num_workers=0, save_step=20000, gradient_accumulation_steps=1)
/content/drive/MyDrive/VPhoBertTagger/vphoberttagger/processor.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data.fillna(method="ffill")
Load dataset Dataset/train_data.txt...: 100% 3364352/3364352 [17:51<00:00, 3139.90it/s]
/content/drive/MyDrive/VPhoBertTagger/vphoberttagger/processor.py:43: FutureWarning: DataFrame.fillna with 'method' is deprecated and w

: 

## Bước 7: Đánh giá Độc lập trên Tập Test (CoNLL Strict Evaluation)

In [ ]:
!python main.py test --data_dir ./Dataset --model_path outputs/best_model.pt

## Bước 8: Tải VnCoreNLP & Dự đoán Thực thể trên Văn bản Mới (Inference)

In [ ]:
# Tải VnCoreNLP jar nếu chưa có
if not os.path.exists("vncorenlp/VnCoreNLP-1.1.1.jar"):
    !mkdir -p vncorenlp/models/wordsegmenter
    !wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.1.1.jar -O vncorenlp/VnCoreNLP-1.1.1.jar
    !wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/vi-vocab -O vncorenlp/models/wordsegmenter/vi-vocab
    !wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/wordsegmenter.rdr -O vncorenlp/models/wordsegmenter/wordsegmenter.rdr
    print("✅ Tải VnCoreNLP thành công!")
else:
    print("✅ VnCoreNLP đã có sẵn!")

In [ ]:
from vphoberttagger.predictor import ViTagger

# Nạp mô hình vừa huấn luyện
tagger = ViTagger(model_path="outputs/best_model.pt")

# Thử nghiệm câu văn bản pháp luật / hành chính thực tế:
sample_text = "Ủy ban nhân dân tỉnh Thái Bình ban hành Nghị định 123/2020/NĐ-CP ngày 15 tháng 10 năm 2020 quy định về hồ sơ đăng ký doanh nghiệp tư nhân với lệ phí 5 triệu đồng."

entities = tagger(sample_text)
print("\n--- KẾT QUẢ TRÍCH XUẤT THỰC THỂ (LEGAL ENTITIES) ---")
for entity, label in entities:
    print(f"{entity:40} -> [{label}]")